# WAXAL ASR — Whisper-small Multilingual Training + Submission

Train `openai/whisper-small` on all 3 WAXAL languages (Lingala, Shona, Luganda) in a single run, then generate a competition submission CSV.

**Requirements:**
- Colab GPU runtime (T4 or A100)
- HuggingFace token with access to `google/WaxalNLP`
- `Test.csv` from the Zindi competition (upload or Google Drive)
- Google Drive (recommended — saves checkpoints across sessions)

## 1. Environment Setup

In [ ]:
import torch

assert torch.cuda.is_available(), "Runtime → Change runtime type → T4 GPU"
gpu_name = torch.cuda.get_device_name(0)
gpu_mem = torch.cuda.get_device_properties(0).total_mem / 1e9
print(f"GPU: {gpu_name} ({gpu_mem:.1f} GB VRAM)")

In [ ]:
import os

repo_url = "https://github.com/Zeusse-Neumbi/google-waxal-asr-challenge.git"
branch = "feature/whisper-baseline"

if not os.path.exists("google-waxal-asr-challenge"):
    !git clone -b {branch} {repo_url}
else:
    !cd google-waxal-asr-challenge && git pull

%cd google-waxal-asr-challenge

In [ ]:
# Install project + dependencies (~3-5 min)
!pip install -e ".[dev,tracking]" -q
!pip install hf_transfer -q

# Verify
!python -c "from waxal_asr.training.trainer import Trainer; print('waxal_asr OK')"

In [ ]:
import shutil
total, used, free = shutil.disk_usage("/")
print(f"Disk: {total / 1e9:.0f} GB total | {used / 1e9:.0f} GB used | {free / 1e9:.0f} GB free")

## 2. Authentication & Storage

Set up HuggingFace token and mount Google Drive for checkpoint persistence.

In [ ]:
import os

# Set HF cache to local disk (faster than Drive for downloads)
os.environ["HF_HOME"] = "/content/hf_cache"
os.environ["HF_DATASETS_CACHE"] = "/content/hf_cache/datasets"

# Load HF token from Colab Secrets
try:
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN")
    os.environ["HUGGING_FACE_HUB_TOKEN"] = hf_token
    os.environ["HF_TOKEN"] = hf_token
    print("HF token loaded from Colab Secrets.")
except Exception:
    hf_token = input("Enter your HuggingFace token: ").strip()
    os.environ["HUGGING_FACE_HUB_TOKEN"] = hf_token
    os.environ["HF_TOKEN"] = hf_token

from huggingface_hub import HfApi
user = HfApi().whoami()
print(f"Logged in as: {user.get('name', 'unknown')}")

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
drive_output = Path("/content/drive/MyDrive/waxal-asr/outputs")
drive_output.mkdir(parents=True, exist_ok=True)
print(f"Checkpoints → {drive_output}")

## 3. Training Configuration

Load `whisper-small.yaml` and apply Colab-specific overrides. The config trains on all 3 languages (lin, sna, lug) via streaming — no dataset materialisation, one batch in RAM at a time.

In [ ]:
from waxal_asr.config import load_config

cfg = load_config("configs/whisper-small.yaml")

# --- Colab overrides ---
# Save to Google Drive
cfg.paths.outputs = drive_output

# Use fp16 (T4 handles fp16 better than bf16)
cfg.model.torch_dtype = "float16"

# Limit training samples per language (set to None for full training)
# 500/language × 3 languages = 1500 total — quick test run
# For full training: set to None and increase max_steps
cfg.dataset.max_train_samples = 500

# Training steps (100 for quick test, 5000 for full training)
cfg.training.max_steps = 100

# Batch size (reduce if OOM)
cfg.training.per_device_train_batch_size = 8
cfg.training.gradient_accumulation_steps = 4

print("=" * 60)
print("Multilingual Whisper-small Training")
print("=" * 60)
print(f"Languages:     {cfg.dataset.languages}")
print(f"Max samples:   {cfg.dataset.max_train_samples} per language")
print(f"Max steps:     {cfg.training.max_steps}")
print(f"Batch size:    {cfg.training.per_device_train_batch_size}")
print(f"Grad accum:    {cfg.training.gradient_accumulation_steps}")
print(f"Learning rate: {cfg.optimizer.lr}")
print(f"LoRA:          {cfg.lora.enabled}")
print(f"Output dir:    {cfg.paths.outputs}")

## 4. Run Training

Trains on all 3 languages interleaved via true streaming. Only one batch in RAM at a time.

**Expected time on T4:** ~15-20 min for 100 steps with 500 samples/language.

In [ ]:
from waxal_asr.utils.seeding import seed_everything
from waxal_asr.training.trainer import Trainer
from waxal_asr.utils.logging import configure_logging

seed_everything(cfg.repro.seed, deterministic=cfg.repro.deterministic)
configure_logging(level=cfg.logging.level)

trainer = Trainer(cfg)
trainer.fit()

## 5. Generate Submission

Transcribe the test set and produce a CSV matching `SampleSubmission.csv` format (columns: `ID,Target`).

**You need `Test.csv` from the Zindi competition.** Either:
- Upload it: run the next cell and use the file picker
- Or place it in Google Drive and update `test_csv_path`

In [ ]:
import pandas as pd
from pathlib import Path

# Option A: Upload Test.csv directly
from google.colab import files
uploaded = files.upload()
test_csv_path = list(uploaded.keys())[0]

# Option B: Use Drive (uncomment and set path)
# test_csv_path = "/content/drive/MyDrive/waxal-asr/Test.csv"

test_df = pd.read_csv(test_csv_path)
print(f"Test IDs: {len(test_df)}")
print(f"Columns: {list(test_df.columns)}")
print(test_df.head())

# Group IDs by language prefix
test_df["lang"] = test_df["ID"].str.split("_").str[0]
id_groups = test_df.groupby("lang")["ID"].apply(list).to_dict()
print(f"\nIDs per language: {[(k, len(v)) for k, v in id_groups.items()]}")

In [ ]:
import torch
import numpy as np
from waxal_asr.models.registry import build_model

# Find the trained checkpoint
checkpoint_dir = drive_output / "whisper-asr-multilingual" / "final"
if not checkpoint_dir.exists():
    # Try checkpoint-N instead
    ckpts = sorted((drive_output / "whisper-asr-multilingual").glob("checkpoint-*"))
    if ckpts:
        checkpoint_dir = ckpts[-1]
    else:
        raise FileNotFoundError(f"No checkpoint found in {drive_output / 'whisper-asr-multilingual'}")

print(f"Using checkpoint: {checkpoint_dir}")

# Build and load the model
model_obj = build_model(cfg.model.model_type, config=cfg)
model_obj.load(checkpoint=str(checkpoint_dir))
model = model_obj.model
processor = model_obj.processor
device = model.device
model.eval()
print("Model loaded.")

In [ ]:
from waxal_asr.data.dataset import load_waxal_dataset
from tqdm import tqdm
import pandas as pd

# Transcribe each language's test samples
all_predictions = {}  # ID -> transcription

for lang, ids in id_groups.items():
    print(f"\nTranscribing {len(ids)} {lang} samples...")

    # Load the test split for this language (streaming)
    test_ds = load_waxal_dataset(
        dataset_id=cfg.dataset.dataset_id,
        language=lang,
        split="test",
        streaming=True,
        sample_rate=cfg.dataset.sample_rate,
    )

    # Stream through and transcribe only matching IDs
    ids_set = set(ids)
    found = 0

    for ex in tqdm(test_ds, desc=f"{lang}"):
        ex_id = str(ex.get("id", ""))
        if ex_id not in ids_set:
            continue

        # Transcribe
        audio_array = np.asarray(ex["audio"]["array"]).flatten()
        audio_tensor = torch.from_numpy(audio_array)
        sr = ex["audio"]["sampling_rate"]

        text = model_obj.transcribe(audio_tensor, sr, max_new_tokens=128)
        all_predictions[ex_id] = text
        found += 1

        if found >= len(ids_set):
            break

    print(f"  Found {found}/{len(ids)} {lang} samples")

print(f"\nTotal transcribed: {len(all_predictions)}")

In [ ]:
from waxal_asr.submission.generator import SubmissionGenerator

# Build submission in the exact order of Test.csv
ids = test_df["ID"].tolist()
targets = [all_predictions.get(id, "") for id in ids]

# Use the project's SubmissionGenerator (auto-increments filename, validates)
generator = SubmissionGenerator(output_dir=drive_output)
submission_path = generator.generate(ids=ids, predictions=targets)

print(f"Submission written: {submission_path}")
print(f"Rows: {len(ids)}")

# Show sample
submission_df = pd.read_csv(submission_path)
print(submission_df.head(10))

In [ ]:
# Validate against SampleSubmission format
from waxal_asr.submission.generator import SubmissionValidator

report = SubmissionValidator.validate(submission_path)
print(f"Valid: {report['passed']}")
print(f"Columns: {report['columns']}")
print(f"Rows: {report['rows']}")
if report["errors"]:
    print(f"Errors: {report['errors']}")

# Sanity check: compare with Test.csv
assert len(submission_df) == len(test_df), f"Row mismatch: {len(submission_df)} vs {len(test_df)}"
assert list(submission_df.columns) == ["ID", "Target"], f"Column mismatch"
assert submission_df["ID"].equals(test_df["ID"]), "ID mismatch with Test.csv"
print("\n✅ Submission validated — matches Test.csv format.")

## 6. Next Steps

- **Full training:** Set `cfg.dataset.max_train_samples = None` and `cfg.training.max_steps = 5000`
- **Stronger model:** Try `whisper-medium` or `whisper-large-v3` (needs A100)
- **Error analysis:** Analyze per-language WER to identify which language needs more data
- **Augmentation:** Enable SpecAugment/noise in config for better generalization